In [1]:
import pyarrow as pa

with pa.memory_map("/home/carol/chaeyeon-kim/alpha26/carla-autopilot-multimodal-dataset-test-00000-of-00049.arrow", "r") as f:
    reader = pa.ipc.open_stream(f)
    table = reader.read_all()

print(table.schema)
print(table.num_rows)
print(table[0])

run_id: string
frame: int32
timestamp: float
image_front: struct<bytes: binary, path: string>
  child 0, bytes: binary
  child 1, path: string
image_front_left: struct<bytes: binary, path: string>
  child 0, bytes: binary
  child 1, path: string
image_front_right: struct<bytes: binary, path: string>
  child 0, bytes: binary
  child 1, path: string
image_rear: struct<bytes: binary, path: string>
  child 0, bytes: binary
  child 1, path: string
seg_front: struct<bytes: binary, path: string>
  child 0, bytes: binary
  child 1, path: string
lidar: list<item: list<item: float>>
  child 0, item: list<item: float>
      child 0, item: float
boxes: list<item: list<item: float>>
  child 0, item: list<item: float>
      child 0, item: float
box_labels: list<item: string>
  child 0, item: string
location_x: float
location_y: float
location_z: float
rotation_pitch: float
rotation_yaw: float
rotation_roll: float
velocity_x: float
velocity_y: float
velocity_z: float
speed_kmh: float
throttle: float


In [2]:
import numpy as np

row = table.to_pydict()
lidar_sample = row["lidar"][0]
lidar_np = np.array(lidar_sample)
print(lidar_np.shape)   # (N, ?) 확인
print(lidar_np[0])      # 첫 번째 포인트

(4856, 4)
[-7.43413773e+01  1.87355903e-12  1.31083908e+01  7.39372909e-01]


In [3]:
import numpy as np

def point_cloud_to_range_view(points, H=64, W=512):
    """
    points: (N, 4) - x, y, z, intensity
    returns: (4, H, W)
    """
    x, y, z, intensity = points[:, 0], points[:, 1], points[:, 2], points[:, 3]

    # 구면 좌표 변환
    r = np.sqrt(x**2 + y**2 + z**2)
    yaw = np.arctan2(y, x)                    # [-π, π]
    pitch = np.arcsin(z / (r + 1e-8))         # [-π/2, π/2]

    # 정규화 → pixel 좌표
    yaw_img   = (yaw   / np.pi + 1) / 2       # [0, 1]
    pitch_img = (pitch / (np.pi/2) + 1) / 2   # [0, 1]

    col = (yaw_img   * (W - 1)).astype(int)
    row = ((1 - pitch_img) * (H - 1)).astype(int)

    col = np.clip(col, 0, W - 1)
    row = np.clip(row, 0, H - 1)

    range_view = np.zeros((4, H, W), dtype=np.float32)
    range_view[0, row, col] = x
    range_view[1, row, col] = y
    range_view[2, row, col] = z
    range_view[3, row, col] = intensity

    return range_view

# 테스트
rv = point_cloud_to_range_view(lidar_np)
print(rv.shape)   # (4, 64, 512)

(4, 64, 512)


In [4]:
from PIL import Image
import io
import numpy as np

# bytes → PIL Image
img_bytes = table.to_pydict()["image_front"][0]["bytes"]
img = Image.open(io.BytesIO(img_bytes))
print(img.size)   # (W, H)
print(img.mode)   # RGB 확인

(800, 600)
RGBA


In [5]:
img_rgb = img.convert("RGB")
print(img_rgb.mode)   # RGB
print(np.array(img_rgb).shape)   # (600, 800, 3)

RGB
(600, 800, 3)


In [6]:
from torchvision import transforms

to_tensor = transforms.ToTensor()
img_tensor = to_tensor(img_rgb)
print(img_tensor.shape)   # (3, 600, 800)

torch.Size([3, 600, 800])


In [7]:
d = table.to_pydict()

print(d["run_id"][:10])
print(d["frame"][:10])

['run_001', 'run_001', 'run_001', 'run_001', 'run_001', 'run_001', 'run_001', 'run_001', 'run_001', 'run_001']
[0, 5, 10, 15, 20, 25, 30, 35, 40, 45]


In [8]:
import pyarrow as pa
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import io
import numpy as np
from torchvision import transforms

to_tensor = transforms.ToTensor()

def point_cloud_to_range_view(points, H=64, W=512):
    x, y, z, intensity = points[:, 0], points[:, 1], points[:, 2], points[:, 3]
    r = np.sqrt(x**2 + y**2 + z**2)
    yaw   = np.arctan2(y, x)
    pitch = np.arcsin(z / (r + 1e-8))
    yaw_img   = (yaw   / np.pi + 1) / 2
    pitch_img = (pitch / (np.pi/2) + 1) / 2
    col = np.clip((yaw_img   * (W - 1)).astype(int), 0, W - 1)
    row = np.clip(((1 - pitch_img) * (H - 1)).astype(int), 0, H - 1)
    rv = np.zeros((4, H, W), dtype=np.float32)
    rv[0, row, col] = x
    rv[1, row, col] = y
    rv[2, row, col] = z
    rv[3, row, col] = intensity
    return rv

class MUVODataset(Dataset):
    def __init__(self, arrow_file, seq_len=4):
        self.seq_len = seq_len
        self.arrow_file = arrow_file

        with pa.memory_map(arrow_file, "r") as f:
            reader = pa.ipc.open_stream(f)
            table = reader.read_all()

        self.data = table.to_pydict()
        self.n = table.num_rows

        # 같은 run_id 안에서 연속 seq_len개 묶기
        self.index = []
        run_ids = self.data["run_id"]
        for i in range(self.n - seq_len + 1):
            if all(run_ids[i+j] == run_ids[i] for j in range(seq_len)):
                self.index.append(i)

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        i = self.index[idx]
        images, lidars, actions, speeds = [], [], [], []

        for j in range(self.seq_len):
            k = i + j
            img = Image.open(io.BytesIO(self.data["image_front"][k]["bytes"])).convert("RGB")
            images.append(to_tensor(img))

            lidar_np = np.array(self.data["lidar"][k])
            lidars.append(torch.tensor(point_cloud_to_range_view(lidar_np)))

            actions.append([self.data["throttle"][k], self.data["steer"][k]])
            speeds.append(self.data["speed_kmh"][k])

        return {
            "image":  torch.stack(images),    # (S, 3, 600, 800)
            "lidar":  torch.stack(lidars),    # (S, 4, 64, 512)
            "action": torch.tensor(actions),  # (S, 2)
            "speed":  torch.tensor(speeds),   # (S,)
        }

# 테스트
arrow_file = "/home/carol/chaeyeon-kim/alpha26/carla-autopilot-multimodal-dataset-test-00000-of-00049.arrow"
ds = MUVODataset(arrow_file, seq_len=4)
print(f"dataset size: {len(ds)}")

sample = ds[0]
for k, v in sample.items():
    print(k, v.shape)

dataset size: 110
image torch.Size([4, 3, 600, 800])
lidar torch.Size([4, 4, 64, 512])
action torch.Size([4, 2])
speed torch.Size([4])


In [9]:
from torch.utils.data import DataLoader

loader = DataLoader(ds, batch_size=2, shuffle=True)
batch = next(iter(loader))

for k, v in batch.items():
    print(k, v.shape)
# image:  (2, 4, 3, 600, 800)
# lidar:  (2, 4, 4, 64, 512)
# action: (2, 4, 2)
# speed:  (2, 4)

image torch.Size([2, 4, 3, 600, 800])
lidar torch.Size([2, 4, 4, 64, 512])
action torch.Size([2, 4, 2])
speed torch.Size([2, 4])


In [10]:
import timm
import torch

# encoder 정의
encoder = timm.create_model(
    'resnet18',
    pretrained=True,
    features_only=True,
    out_indices=[2, 3, 4]
)
encoder.eval()

# (B, S, 3, 600, 800) → (B*S, 3, 600, 800)
B, S = 2, 4
image = batch["image"]                        # (2, 4, 3, 600, 800)
image_flat = image.view(B*S, 3, 600, 800)    # (8, 3, 600, 800)

xs = encoder(image_flat)
for i, x in enumerate(xs):
    print(f"scale {i}: {x.shape}")

scale 0: torch.Size([8, 128, 75, 100])
scale 1: torch.Size([8, 256, 38, 50])
scale 2: torch.Size([8, 512, 19, 25])


In [11]:
import torch.nn as nn
import torch.nn.functional as F
from typing import List

class FPNDecoder(nn.Module):
    def __init__(self, feature_info, out_channels=128):
        super().__init__()
        # 가장 깊은 feature부터 시작
        self.conv1 = nn.Sequential(
            nn.Conv2d(feature_info[2], out_channels, 3, 1, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(True),
        )
        self.skip_convs = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(feature_info[i], out_channels, 3, 1, 1, bias=False),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(True),
            )
            for i in [1, 0]  # scale2, scale1 순서
        ])
        self.out_channels = out_channels

    def forward(self, xs: List[torch.Tensor]) -> torch.Tensor:
        x = self.conv1(xs[2])  # (B*S, out_channels, 19, 25)

        for i, conv in enumerate(self.skip_convs):
            size = xs[1 - i].shape[-2:]
            x = conv(xs[1 - i]) + F.interpolate(x, size=size, mode='bilinear', align_corners=False)

        return x  # (B*S, out_channels, 75, 100)

# 테스트
feature_info = [128, 256, 512]
fpn = FPNDecoder(feature_info, out_channels=128)
fpn.eval()

with torch.no_grad():
    Fc = fpn(xs)
    print(f"Fc: {Fc.shape}")  # (8, 128, 75, 100)

Fc: torch.Size([8, 128, 75, 100])


In [12]:
# LiDAR encoder (range-view 4채널 입력)
lidar_encoder = timm.create_model(
    'resnet18',
    pretrained=True,
    features_only=True,
    out_indices=[2, 3, 4],
    in_chans=4          # RGB 3채널 → LiDAR 4채널
)
lidar_encoder.eval()

lidar_fpn = FPNDecoder([128, 256, 512], out_channels=128)
lidar_fpn.eval()

# (B, S, 4, 64, 512) → (B*S, 4, 64, 512)
lidar_flat = batch["lidar"].view(B*S, 4, 64, 512)

with torch.no_grad():
    lidar_xs = lidar_encoder(lidar_flat)
    for i, x in enumerate(lidar_xs):
        print(f"lidar scale {i}: {x.shape}")
    
    FL = lidar_fpn(lidar_xs)
    print(f"FL: {FL.shape}")

lidar scale 0: torch.Size([8, 128, 8, 64])
lidar scale 1: torch.Size([8, 256, 4, 32])
lidar scale 2: torch.Size([8, 512, 2, 16])
FL: torch.Size([8, 128, 8, 64])


In [13]:
BS, C, H, W = Fc.shape
Fc = Fc.view(B, S, C, H, W)                  # (B, S, C, 75, 100)

BS, C, H, W = FL.shape
FL = FL.view(B, S, C, H, W)                  # (B, S, C, 8, 64)

Fc = Fc.permute(1, 0, 2, 3, 4).contiguous()  # (S, B, C, 75, 100)
FL = FL.permute(1, 0, 2, 3, 4).contiguous()  # (S, B, C, 8, 64)

# camera: (75, 100) -> (15, 20)
S_cam, B_cam, C_cam, H_cam, W_cam = Fc.shape
Fc = F.adaptive_avg_pool2d(
    Fc.reshape(S_cam * B_cam, C_cam, H_cam, W_cam),
    (15, 20)
).view(S_cam, B_cam, C_cam, 15, 20)

# lidar: (8, 64) -> (8, 16)
S_lidar, B_lidar, C_lidar, H_lidar, W_lidar = FL.shape
FL = F.adaptive_avg_pool2d(
    FL.reshape(S_lidar * B_lidar, C_lidar, H_lidar, W_lidar),
    (8, 16)
).view(S_lidar, B_lidar, C_lidar, 8, 16)

print("Fc:", Fc.shape)   # (4, 2, 128, 15, 20)
print("FL:", FL.shape)   # (4, 2, 128, 8, 16)

Fc: torch.Size([4, 2, 128, 15, 20])
FL: torch.Size([4, 2, 128, 8, 16])


In [14]:
import math
## num_pos_feats = C // 2
class PositionEmbeddingSine(nn.Module):
    def __init__(self, num_pos_feats=64, temperature=10000, normalize=False, scale=None):
        super().__init__()
        self.num_pos_feats = num_pos_feats
        self.temperature = temperature
        self.normalize = normalize
        if scale is not None and normalize is False:
            raise ValueError("normalize should be True if scale is passed")
        if scale is None:
            scale = 2 * math.pi
        self.scale = scale
    # 입력 feature의 크기(H,W)에 맞는 위치 embedding을 계산하는 함수 
    
    def forward(self, tensor):
        x = tensor
        B, C, h, w = x.shape

        not_mask = torch.ones((B, h, w), device=x.device)
        y_embed = not_mask.cumsum(1, dtype=torch.float32)
        x_embed = not_mask.cumsum(2, dtype=torch.float32)

        if self.normalize:
            eps = 1e-6
            y_embed = y_embed / (y_embed[:, -1:, :] + eps) * self.scale
            x_embed = x_embed / (x_embed[:, :, -1:] + eps) * self.scale

        dim_t = torch.arange(self.num_pos_feats, dtype=torch.float32, device=x.device)
        dim_t = self.temperature ** (2 * (dim_t // 2) / self.num_pos_feats)

        pos_x = x_embed[:, :, :, None] / dim_t   # (B, h, w, num_pos_feats)
        pos_y = y_embed[:, :, :, None] / dim_t

        pos_x = torch.stack(
            (pos_x[:, :, :, 0::2].sin(), pos_x[:, :, :, 1::2].cos()), dim=4
        ).flatten(3)

        pos_y = torch.stack(
            (pos_y[:, :, :, 0::2].sin(), pos_y[:, :, :, 1::2].cos()), dim=4
        ).flatten(3)

        pos = torch.cat((pos_y, pos_x), dim=3).permute(0, 3, 1, 2)  # (B, C, h, w)
        return pos


class SensorFusionTransformer(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.channels = channels

        self.position_encode = PositionEmbeddingSine(
            num_pos_feats=channels // 2,
            normalize=True
        )

        # sensor type embedding: 0=camera, 1=lidar
        self.type_embedding = nn.Parameter(torch.zeros(2, channels))

        self.encoder_layer = nn.TransformerEncoderLayer(
            d_model=channels,
            nhead=8,
            dropout=0.1,
            batch_first=False   # default
        )
        self.transformer_encoder = nn.TransformerEncoder(self.encoder_layer, num_layers=6)

    def feature_to_tokens(self, x):
        # x: (B, C, H, W)
        # return: (H*W, B, C)
        return x.flatten(2).permute(2, 0, 1)

    def add_pos_and_type(self, x, sensor_type_idx):
        
        tokens = self.feature_to_tokens(x)  # (H*W, B, C)

        pos = self.position_encode(x)       # (B, C, H, W)
        pos_tokens = self.feature_to_tokens(pos)  # (H*W, B, C)

        NHW, B, C = tokens.shape

        type_embed = self.type_embedding[sensor_type_idx].view(1, 1, C)  # (1, 1, C)
        type_embed = type_embed.expand(NHW, B, C)            # (H*W, B, C)

        tokens = tokens + pos_tokens + type_embed
        return tokens

    def forward(self, cam_feat, lidar_feat):
        # cam_feat:   (B, C, Hc, Wc)
        # lidar_feat: (B, C, Hl, Wl)

        cam_tokens = self.add_pos_and_type(cam_feat, sensor_type_idx=0)
        lidar_tokens = self.add_pos_and_type(lidar_feat, sensor_type_idx=1)

        # sequence dimension으로 concat
        fused_tokens = torch.cat([cam_tokens, lidar_tokens], dim=0)
        # shape: (Nc + Nl, B, C)

        out = self.transformer_encoder(fused_tokens)
        return out
        # Transformer encoder를 통과한 뒤의 최종 토큰들

In [15]:
class RepresentationModel(nn.Module):
    def __init__(self, in_channels, latent_dim):
        super().__init__()
        self.latent_dim = latent_dim
        self.min_std = 0.1

        self.module = nn.Sequential(
            nn.Linear(in_channels, in_channels),
            nn.LeakyReLU(True),
            nn.Linear(in_channels, 2 * self.latent_dim),
        )

    def forward(self, x):
        def sigmoid2(tensor: torch.Tensor, min_value: float) -> torch.Tensor:
            return 2 * torch.sigmoid(tensor / 2) + min_value

        mu_log_sigma = self.module(x)
        mu, log_sigma = torch.split(mu_log_sigma, self.latent_dim, dim=-1)
        sigma = sigmoid2(log_sigma, self.min_std)
        return mu, sigma


class RSSM(nn.Module):
    def __init__(
        self,
        embedding_dim,
        action_dim,
        hidden_state_dim,
        state_dim,
        action_latent_dim,
        receptive_field,
        use_dropout=False,
        dropout_probability=0.0,
    ):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.hidden_state_dim = hidden_state_dim
        self.action_latent_dim = action_latent_dim
        self.receptive_field = receptive_field
        self.use_dropout = use_dropout
        self.dropout_probability = dropout_probability

        self.pre_gru_net = nn.Sequential(
            nn.Linear(state_dim, hidden_state_dim),
            nn.LeakyReLU(True),
        )

        self.recurrent_model = nn.GRUCell(
            input_size=hidden_state_dim,
            hidden_size=hidden_state_dim,
        )

        self.posterior_action_module = nn.Sequential(
            nn.Linear(action_dim, self.action_latent_dim),
            nn.LeakyReLU(True),
        )

        self.posterior = RepresentationModel(
            in_channels=hidden_state_dim + embedding_dim + self.action_latent_dim,
            latent_dim=state_dim,
        )

        self.prior_action_module = nn.Sequential(
            nn.Linear(action_dim, self.action_latent_dim),
            nn.LeakyReLU(True),
        )

        self.prior = RepresentationModel(
            in_channels=hidden_state_dim + self.action_latent_dim,
            latent_dim=state_dim,
        )

        self.active_inference = False

    def forward(self, input_embedding, action, use_sample=True, policy=None):
        output = {
            "prior": [],
            "posterior": [],
        }

        batch_size, sequence_length, _ = input_embedding.shape
        h_t = input_embedding.new_zeros((batch_size, self.hidden_state_dim))
        sample_t = input_embedding.new_zeros((batch_size, self.state_dim))

        for t in range(sequence_length):
            if t == 0:
                action_t = torch.zeros_like(action[:, 0])
            else:
                action_t = action[:, t - 1]

            output_t = self.observe_step(
                h_t,
                sample_t,
                action_t,
                input_embedding[:, t],
                use_sample=use_sample,
                policy=policy,
            )

            use_prior = (
                self.training
                and self.use_dropout
                and torch.rand(1).item() < self.dropout_probability
                and t > 0
            )

            if use_prior:
                sample_t = output_t["prior"]["sample"]
            else:
                sample_t = output_t["posterior"]["sample"]

            h_t = output_t["prior"]["hidden_state"]

            for key, value in output_t.items():
                output[key].append(value)

        return self.stack_list_of_dict_tensor(output, dim=1)

    def observe_step(self, h_t, sample_t, action_t, embedding_t, use_sample=True, policy=None):
        imagine_output = self.imagine_step(h_t, sample_t, action_t, use_sample, policy=policy)

        latent_action_t = self.posterior_action_module(action_t)
        posterior_mu_t, posterior_sigma_t = self.posterior(
            torch.cat([imagine_output["hidden_state"], embedding_t, latent_action_t], dim=-1)
        )

        sample_t = self.sample_from_distribution(
            posterior_mu_t, posterior_sigma_t, use_sample=use_sample
        )

        posterior_output = {
            "hidden_state": imagine_output["hidden_state"],
            "sample": sample_t,
            "mu": posterior_mu_t,
            "sigma": posterior_sigma_t,
        }

        return {
            "prior": imagine_output,
            "posterior": posterior_output,
        }

    def imagine_step(self, h_t, sample_t, action_t, use_sample=True, policy=None):
        if self.active_inference:
            action_t = policy(torch.cat([h_t, sample_t], dim=-1))

        latent_action_t = self.prior_action_module(action_t)

        input_t = self.pre_gru_net(sample_t)
        h_t = self.recurrent_model(input_t, h_t)
        prior_mu_t, prior_sigma_t = self.prior(torch.cat([h_t, latent_action_t], dim=-1))
        sample_t = self.sample_from_distribution(prior_mu_t, prior_sigma_t, use_sample=use_sample)

        return {
            "hidden_state": h_t,
            "sample": sample_t,
            "mu": prior_mu_t,
            "sigma": prior_sigma_t,
        }

    @staticmethod
    def sample_from_distribution(mu, sigma, use_sample):
        sample = mu
        if use_sample:
            noise = torch.randn_like(sample)
            sample = sample + sigma * noise
        return sample

    @staticmethod
    def stack_list_of_dict_tensor(output, dim=1):
        new_output = {}
        for outter_key, outter_value in output.items():
            if len(outter_value) > 0:
                new_output[outter_key] = {}
                for inner_key in outter_value[0].keys():
                    new_output[outter_key][inner_key] = torch.stack(
                        [x[inner_key] for x in outter_value],
                        dim=dim,
                    )
        return new_output


class Model(nn.Module):
    def __init__(self, cfg, embedding_n_channels, transformer_encoder):
        super().__init__()
        self.cfg = cfg
        self.transformer_encoder = transformer_encoder

        self.receptive_field = self.cfg.RECEPTIVE_FIELD
        if self.cfg.MODEL.TRANSITION.ENABLED:
            self.rssm = RSSM(
                embedding_dim=embedding_n_channels,
                action_dim=self.cfg.MODEL.ACTION_DIM,
                hidden_state_dim=self.cfg.MODEL.TRANSITION.HIDDEN_STATE_DIM,
                state_dim=self.cfg.MODEL.TRANSITION.STATE_DIM,
                action_latent_dim=self.cfg.MODEL.TRANSITION.ACTION_LATENT_DIM,
                receptive_field=self.receptive_field,
                use_dropout=self.cfg.MODEL.TRANSITION.USE_DROPOUT,
                dropout_probability=self.cfg.MODEL.TRANSITION.DROPOUT_PROBABILITY,
            )

    def forward(self, cam_feat_seq, lidar_feat_seq, action):
        """
        cam_feat_seq:   (S, B, C, Hc, Wc)
        lidar_feat_seq: (S, B, C, Hl, Wl)
        action:         (B, S, action_dim)
        """
        embeddings = []
        S = cam_feat_seq.shape[0]

        for t in range(S):
            out = self.transformer_encoder(cam_feat_seq[t], lidar_feat_seq[t])  # (N_tokens, B, C)
            feat = out.mean(dim=0)  # (B, C)
            embeddings.append(feat)

        embedding_seq = torch.stack(embeddings, dim=1)  # (B, S, C)

        rssm_output = self.rssm(
            input_embedding=embedding_seq,
            action=action,
            use_sample=True,
            policy=None,
        )
        return rssm_output

In [16]:
def main():
    import torch
    from torch.utils.data import DataLoader
    import timm

    # =========================
    # 1. Dataset / Loader
    # =========================
    arrow_file = "/home/carol/chaeyeon-kim/alpha26/carla-autopilot-multimodal-dataset-test-00000-of-00049.arrow"
    ds = MUVODataset(arrow_file, seq_len=4)
    loader = DataLoader(ds, batch_size=2, shuffle=True)

    batch = next(iter(loader))

    image = batch["image"]      # (B, S, 3, 600, 800)
    lidar = batch["lidar"]      # (B, S, 4, 64, 512)
    action = batch["action"]    # (B, S, 2)

    B, S = image.shape[0], image.shape[1]

    # =========================
    # 2. Camera Encoder + FPN
    # =========================
    encoder = timm.create_model(
        'resnet18',
        pretrained=True,
        features_only=True,
        out_indices=[2, 3, 4]
    ).eval()

    fpn = FPNDecoder([128, 256, 512], out_channels=128).eval()

    image_flat = image.view(B*S, 3, 600, 800)

    with torch.no_grad():
        xs = encoder(image_flat)
        Fc = fpn(xs)   # (B*S, 128, 75, 100)

    # =========================
    # 3. LiDAR Encoder + FPN
    # =========================
    lidar_encoder = timm.create_model(
        'resnet18',
        pretrained=True,
        features_only=True,
        out_indices=[2, 3, 4],
        in_chans=4
    ).eval()

    lidar_fpn = FPNDecoder([128, 256, 512], out_channels=128).eval()

    lidar_flat = lidar.view(B*S, 4, 64, 512)

    with torch.no_grad():
        lidar_xs = lidar_encoder(lidar_flat)
        FL = lidar_fpn(lidar_xs)   # (B*S, 128, 8, 64)

    # =========================
    # 4. reshape → (S, B, C, H, W)
    # =========================
    _, C, H, W = Fc.shape
    Fc = Fc.view(B, S, C, H, W)

    _, C, H, W = FL.shape
    FL = FL.view(B, S, C, H, W)

    Fc = Fc.permute(1, 0, 2, 3, 4).contiguous()
    FL = FL.permute(1, 0, 2, 3, 4).contiguous()

    # =========================
    # 5. Downsampling
    # =========================
    import torch.nn.functional as F

    S_cam, B_cam, C_cam, H_cam, W_cam = Fc.shape
    Fc = F.adaptive_avg_pool2d(
        Fc.reshape(S_cam * B_cam, C_cam, H_cam, W_cam),
        (15, 20)
    ).view(S_cam, B_cam, C_cam, 15, 20)

    S_lidar, B_lidar, C_lidar, H_lidar, W_lidar = FL.shape
    FL = F.adaptive_avg_pool2d(
        FL.reshape(S_lidar * B_lidar, C_lidar, H_lidar, W_lidar),
        (8, 16)
    ).view(S_lidar, B_lidar, C_lidar, 8, 16)

    # =========================
    # 6. Transformer
    # =========================
    transformer = SensorFusionTransformer(channels=128)

    # =========================
    # 7. Model (RSSM 포함)
    # =========================
    class DummyCfg:
        RECEPTIVE_FIELD = 4

        class MODEL:
            ACTION_DIM = 2

            class TRANSITION:
                ENABLED = True
                HIDDEN_STATE_DIM = 128
                STATE_DIM = 64
                ACTION_LATENT_DIM = 32
                USE_DROPOUT = False
                DROPOUT_PROBABILITY = 0.0

    cfg = DummyCfg()

    model = Model(cfg, embedding_n_channels=128, transformer_encoder=transformer)

    # =========================
    # 8. Forward
    # =========================
    output = model(
        cam_feat_seq=Fc,
        lidar_feat_seq=FL,
        action=action
    )

    # =========================
    # 9. 결과 출력
    # =========================
    print("=== OUTPUT ===")
    for k, v in output.items():
        print(k)
        for kk, vv in v.items():
            print(f"  {kk}: {vv.shape}")


if __name__ == "__main__":
    main()

=== OUTPUT ===
prior
  hidden_state: torch.Size([2, 4, 128])
  sample: torch.Size([2, 4, 64])
  mu: torch.Size([2, 4, 64])
  sigma: torch.Size([2, 4, 64])
posterior
  hidden_state: torch.Size([2, 4, 128])
  sample: torch.Size([2, 4, 64])
  mu: torch.Size([2, 4, 64])
  sigma: torch.Size([2, 4, 64])
